## Feature engineering for monthly demand forecasting

In [1]:
# Import Library

from __future__ import annotations

import os
import numpy as np
import pandas as pd

# -----------------------------
# Configuration
# -----------------------------
ERP_PATH = "data/cleaned/erp_cleaned.csv"
EXT_PATH = "data/cleaned/external_cleaned.csv"
OUT_PATH = "data/training/training_data_final.csv"

# Forecasting setup
FORECAST_HORIZON = 1   # 1 = next month
CREATE_TARGETS = True  # creates y_qty and y_revenue

# Feature config
INTERNAL_LAGS = (1, 2, 3, 6, 12)
INTERNAL_WINDOWS = (3, 6, 12)
EXTERNAL_LAGS = (1, 2, 3)
EXTERNAL_WINDOWS = (3, 6)

# Missing value handling
IMPUTE_EXTERNAL = False

def month_start(s: pd.Series) -> pd.Series:
    s = pd.to_datetime(s, errors="coerce")
    return s.dt.to_period("M").dt.to_timestamp(how="start")

def weighted_avg(values: pd.Series, weights: pd.Series) -> float:
    w = weights.to_numpy(dtype=float)
    v = values.to_numpy(dtype=float)
    denom = np.nansum(w)
    if denom == 0:
        return np.nan
    return float(np.nansum(v * w) / denom)

def add_time_features(df: pd.DataFrame, date_col: str = "Date") -> pd.DataFrame:
    dt = pd.to_datetime(df[date_col])
    df["Year"] = dt.dt.year
    df["Month"] = dt.dt.month
    df["Quarter"] = dt.dt.quarter
    df["month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)
    return df

def add_external_features(ext: pd.DataFrame, value_cols: list[str]) -> pd.DataFrame:
    ext = ext.sort_values("Date").reset_index(drop=True)
    ext = ext.groupby("Date", as_index=False).first()

    for c in value_cols:
        ext[f"{c}_missing"] = ext[c].isna().astype("int8")
        for lag in EXTERNAL_LAGS:
            ext[f"{c}_lag{lag}"] = ext[c].shift(lag)

        base = ext[c].shift(1)
        for w in EXTERNAL_WINDOWS:
            ext[f"{c}_rollmean{w}"] = base.rolling(w, min_periods=1).mean()
            ext[f"{c}_rollstd{w}"] = base.rolling(w, min_periods=2).std()
    return ext

def densify_monthly_panel(
    monthly: pd.DataFrame,
    key_cols: list[str],
    date_col: str = "Date",
    target_cols: tuple[str, ...] = ("ordered_qty", "order_amount"),
    static_cols: tuple[str, ...] = ("customer_name", "product_group", "product_name"),
) -> pd.DataFrame:
    monthly = monthly.copy()
    monthly[date_col] = month_start(monthly[date_col])

    min_d, max_d = monthly[date_col].min(), monthly[date_col].max()
    full_months = pd.date_range(min_d, max_d, freq="MS")

    keys = (
        monthly.sort_values(key_cols + [date_col])
               .groupby(key_cols, as_index=False)
               .agg({c: "first" for c in static_cols})
    )

    grid = keys.assign(_k=1).merge(
        pd.DataFrame({date_col: full_months, "_k": 1}),
        on="_k"
    ).drop(columns=["_k"])

    merged = grid.merge(monthly, on=key_cols + list(static_cols) + [date_col], how="left")

    for t in target_cols:
        merged[f"{t}_observed"] = (~merged[t].isna()).astype("int8")
        merged[t] = merged[t].fillna(0.0)

    if "unit_price" in merged.columns:
        merged["unit_price_observed"] = (~merged["unit_price"].isna()).astype("int8")
        merged["unit_price"] = (
            merged.sort_values(key_cols + [date_col])
                  .groupby(key_cols)["unit_price"]
                  .ffill()
        )
        merged["unit_price_missing"] = merged["unit_price"].isna().astype("int8")

    return merged

def add_internal_features(df: pd.DataFrame, key_cols: list[str]) -> pd.DataFrame:
    df = df.sort_values(key_cols + ["Date"]).reset_index(drop=True)
    g = df.groupby(key_cols, group_keys=False)

    for tgt in ("ordered_qty", "order_amount"):
        for lag in INTERNAL_LAGS:
            df[f"{tgt}_lag{lag}"] = g[tgt].shift(lag)

        base = g[tgt].shift(1)
        for w in INTERNAL_WINDOWS:
            df[f"{tgt}_rollmean{w}"] = base.transform(lambda x: x.rolling(w, min_periods=1).mean())
            df[f"{tgt}_rollstd{w}"] = base.transform(lambda x: x.rolling(w, min_periods=2).std())

        lag12 = g[tgt].shift(12)
        df[f"{tgt}_yoy_change"] = np.where(lag12.abs() > 0, (df[tgt] - lag12) / lag12.abs(), np.nan)
        df[f"{tgt}_lag12_is_zero"] = (lag12 == 0).astype("int8")

    df["qty_change_1_3"] = df["ordered_qty_lag1"] / df["ordered_qty_lag3"].replace(0, np.nan)
    df["qty_change_3_6"] = df["ordered_qty_lag3"] / df["ordered_qty_lag6"].replace(0, np.nan)

    first_month = g["Date"].transform("min")
    df["months_since_first"] = (
        (df["Date"].dt.year - first_month.dt.year) * 12 +
        (df["Date"].dt.month - first_month.dt.month)
    ).astype(int)

    df = df.replace([np.inf, -np.inf], np.nan)
    return df

def create_targets(df: pd.DataFrame, key_cols: list[str], horizon: int = 1) -> pd.DataFrame:
    df = df.sort_values(key_cols + ["Date"]).reset_index(drop=True)
    g = df.groupby(key_cols, group_keys=False)

    df["y_qty"] = g["ordered_qty"].shift(-horizon)
    df["y_revenue"] = g["order_amount"].shift(-horizon)

    df = df[df["y_qty"].notna()].copy()
    return df

def main() -> None:
    erp = pd.read_csv(ERP_PATH, low_memory=False)
    external = pd.read_csv(EXT_PATH, low_memory=False)

    erp["Date"] = month_start(erp["Date"])
    external["Date"] = month_start(external["Date"])

    def agg_block(g: pd.DataFrame) -> pd.Series:
        return pd.Series({
            "ordered_qty": g["ordered_qty"].sum(),
            "order_amount": g["order_amount"].sum(),
            "unit_price": weighted_avg(g["unit_price"], g["ordered_qty"]),
            "customer_name": g["customer_name"].iloc[0],
            "product_group": g["product_group"].iloc[0],
            "product_name": g["product_name"].iloc[0],
        })

    erp_monthly = (
        erp.groupby(["customer_id", "product_id", "Date"], as_index=False)
           .apply(agg_block)
           .reset_index(drop=True)
    )

    panel = densify_monthly_panel(
        erp_monthly,
        key_cols=["customer_id", "product_id"],
        date_col="Date",
        target_cols=("ordered_qty", "order_amount"),
        static_cols=("customer_name", "product_group", "product_name"),
    )

    external_cols = [
        "Food_Price_Index",
        "Electricity_Price",
        "ECB_Inflation",
        "ECB_Interest_Rate",
        "FEDFUNDS_Value",
        "Purchase_Index",
        "Total_New_Orders_Value",
        "Steel_Price",
    ]

    is_real_cols = [c for c in external.columns if c.endswith("_is_real")]
    for c in is_real_cols:
        external[c] = external[c].fillna(0).astype("int8")

    external_fe = add_external_features(external, external_cols)

    if IMPUTE_EXTERNAL:
        for c in external_cols:
            external_fe[c] = external_fe[c].ffill()

    df = panel.merge(external_fe, on="Date", how="left")
    df = add_time_features(df, "Date")
    df = add_internal_features(df, key_cols=["customer_id", "product_id"])

    if CREATE_TARGETS:
        df = create_targets(df, key_cols=["customer_id", "product_id"], horizon=FORECAST_HORIZON)

    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    df.to_csv(OUT_PATH, index=False)

    print(f"Saved: {OUT_PATH}")
    print("Final shape:", df.shape)

if __name__ == "__main__":
    main()


C:\Users\chonchol\AppData\Local\Temp\ipykernel_4280\3784100465.py:164: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(agg_block)


Saved: data/training/training_data_final.csv
Final shape: (330939, 129)
